# N1 solution. Guided activity of Notebook 1, solved

**Predicting Flash Floods in Working Landscapes**, Module 1, Quantitative Precipitation Estimation (QPE)
CUAHSI Water Science Exchange 2026

This short notebook holds the completed activity of Notebook 1 (section 9), with its outputs. It rebuilds only what the activity needs, so it runs on its own in about ten seconds. The full notebook with every section executed is `01_where_to_find_qpe_data_solution.ipynb`.

The task. Pick another hour of the storm and another gauge, read the three MRMS hourly products at the gauge location and compare them with the gauge.

## 0. Setup

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path

REPO_URL = "https://github.com/AHWALab/Water-Science-Exchange-Workshop.git"
BRANCH = "module-1-qpe"

# Step 1. If the helper file is not next to this notebook, we are in Colab. Copy the branch (public, no account needed).
if not Path("qpe_tools.py").exists():
    if not Path("Water-Science-Exchange-Workshop").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL, "Water-Science-Exchange-Workshop"], check=True)
    os.chdir("Water-Science-Exchange-Workshop/notebooks")
sys.path.insert(0, os.getcwd())

# Step 2. Install what is missing. Colab lacks the GRIB2 reader and the map library. About one minute the first time.
missing = [m for m in ["eccodes", "cfgrib", "cartopy", "s3fs", "h5netcdf"] if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print("Working folder:", os.getcwd(), "| installed:", ", ".join(missing) if missing else "nothing, all packages present")

Working folder: /home/claude/m1/notebooks | installed: nothing, all packages present


## 1. What the activity needs from Notebook 1

The imports, the region, and the ISU Soil Moisture station table of section 6a.

In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import xarray as xr
from qpe_tools import *

MIDWEST = (-98.0, -90.0, 40.0, 45.0)                 # west, east, south, north
isusm = iem_stations("ISUSM", MIDWEST)              # ISU Soil Moisture Network, tipping buckets
asos = pd.concat([iem_stations(n, MIDWEST) for n in ["IA_ASOS", "SD_ASOS", "MN_ASOS", "NE_ASOS"]])
print(len(isusm), "ISUSM stations and", len(asos), "airports in the box")

27 ISUSM stations and 124 airports in the box


## 2. The activity, solved

Inwood (CSII4) at 02:00 UTC on June 22. Change `MY_HOUR` and `MY_STATION` to try another case. For an airport, replace `iem_isusm_hourly` with `iem_asos_hourly` in step 4 and look the station up in `asos` instead of `isusm`.

In [3]:
#################### Your Code Goes Here #########################
# Activity, step 1. Choose an hour (UTC, end of the hour) and a station.
MY_HOUR = "2024-06-22 02:00"
MY_STATION = "CSII4"             # ISUSM Inwood

# Activity, step 2. Find the coordinates of the station in the ISUSM station table (isusm, section 6a).
row = isusm[isusm.station == MY_STATION].iloc[0]
my_lon, my_lat = row.lon, row.lat
print(row["name"], my_lon, my_lat)
###################################################################

Inwood - Calico Skies (Vineyard) -96.504251 43.310471


In [4]:
#################### Your Code Goes Here #########################
# Activity, step 3. Read the three MRMS hourly products for MY_HOUR over the Midwest box
#                   and take the value of the grid cell nearest to the station.
values = {}
for product in ["RadarOnly", "Pass1", "Pass2"]:
    field = read_mrms(product, MY_HOUR, MIDWEST)
    values[product] = float(field.sel(lon=my_lon, lat=my_lat, method="nearest"))
values
###################################################################

{'RadarOnly': 43.900001525878906,
 'Pass1': 48.400001525878906,
 'Pass2': 46.900001525878906}

In [5]:
#################### Your Code Goes Here #########################
# Activity, step 4. Pull the gauge value for the same hour (iem_isusm_hourly or iem_asos_hourly)
#                   and compare it with the three products. Which product is closest, and why?
gauge = iem_isusm_hourly([MY_STATION], pd.Timestamp(MY_HOUR) - pd.Timedelta(hours=1), MY_HOUR)
gauge_mm = float(gauge.mm.iloc[-1])
print(f"gauge {MY_STATION}: {gauge_mm:.1f} mm")
for product, v in values.items():
    print(f"{LABELS[product]:22s} {v:6.1f} mm   difference {v - gauge_mm:+6.1f} mm")
###################################################################

gauge CSII4: 60.4 mm
Radar only               43.9 mm   difference  -16.5 mm
Multi-sensor Pass 1      48.4 mm   difference  -12.0 mm
Multi-sensor Pass 2      46.9 mm   difference  -13.5 mm


Reading the result. The gauge correction moves the radar value toward the gauges that reported in time. Whether it moves toward this particular gauge depends on whether this gauge was among them, which the gauge influence index of Notebook 2 shows. A difference of a few millimetres in one hour is within the gauge versus grid cell sampling difference. A difference of a factor of two is not.